# Week 3 Internship — Real-World Dataset Analysis Project
**Organization:** Yuva  
**Dataset:** UCI Adult / Census Income dataset  
**Purpose:** Explore, clean, analyze, visualize, and document a public dataset.

> Run cells from top to bottom. This notebook loads a public dataset, so an internet connection may be required. If dataset loading is unavailable in your environment, download the Adult dataset from the UCI Machine Learning Repository and adjust the file path below. This project uses public educational data, not Yuva's internal data.

## 1. Imports and environment

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Seaborn:", sns.__version__)
sns.set_theme()

## 2. Load the dataset

In [ ]:
# Try the UCI repository's ucimlrepo package first.
# If it is not installed, use the raw public data file as a fallback.
try:
    from ucimlrepo import fetch_ucirepo
    adult = fetch_ucirepo(id=2)
    df = adult.data.features.copy()
    if adult.data.targets is not None:
        target = adult.data.targets.copy()
        target.columns = ["income"]
        df["income"] = target["income"].values
    print("Loaded dataset using ucimlrepo.")
except Exception as e:
    print("ucimlrepo loading unavailable:", repr(e))
    print("Trying public UCI raw data file...")
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    columns = [
        "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
        "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
        "hours_per_week", "native_country", "income"
    ]
    df = pd.read_csv(url, header=None, names=columns, na_values="?", skipinitialspace=True)
    print("Loaded dataset from UCI raw data file.")

display(df.head(10))
print("Shape:", df.shape)
print("\nColumn types:")
display(df.dtypes.to_frame("dtype"))

## 3. Understand variables and data quality

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))

print("Duplicate rows:", df.duplicated().sum())

print("\nNumeric summary:")
display(df.describe(include=[np.number]).T)

print("\nCategorical summary:")
display(df.describe(include=["object", "category"]).T)

## 4. Clean and preprocess

In [ ]:
clean = df.copy()

# Normalize text values and standardize missing markers.
for col in clean.select_dtypes(include=["object", "string"]).columns:
    clean[col] = clean[col].astype("string").str.strip()
    clean[col] = clean[col].replace({"?": pd.NA, "": pd.NA})

# Convert numeric-looking fields safely if present.
numeric_candidates = [
    "age", "fnlwgt", "education_num", "capital_gain",
    "capital_loss", "hours_per_week"
]
for col in numeric_candidates:
    if col in clean.columns:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")

print("Missing values after standardizing markers:")
display(clean.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("Duplicate rows before removal:", clean.duplicated().sum())

# Remove exact duplicate records and report how many were removed.
before = len(clean)
clean = clean.drop_duplicates().copy()
print("Duplicate rows removed:", before - len(clean))

# Keep rows with missing values for descriptive analysis, but create a complete-case copy
# for analyses requiring selected fields. This avoids silently imputing every variable.
analysis_cols = [c for c in ["age", "hours_per_week", "income"] if c in clean.columns]
analysis_df = clean.dropna(subset=analysis_cols).copy()
print("Cleaned dataset shape:", clean.shape)
print("Rows complete for core analysis:", analysis_df.shape[0])

## 5. Exploratory analysis: categories and grouped statistics

In [ ]:
if "income" in clean.columns:
    print("Income category counts:")
    display(clean["income"].value_counts(dropna=False).to_frame("count"))
    print("Income category proportions:")
    display(clean["income"].value_counts(normalize=True, dropna=False).mul(100).round(2).to_frame("percent"))

group_cols = [c for c in ["age", "hours_per_week", "education_num", "capital_gain", "capital_loss"] if c in clean.columns]
if "income" in clean.columns and group_cols:
    display(clean.groupby("income", dropna=False)[group_cols].agg(["mean", "median", "std"]).round(2))

## 6. Visualizations

In [ ]:
if "income" in clean.columns:
    plt.figure(figsize=(7, 4))
    order = clean["income"].value_counts().index
    sns.countplot(data=clean, x="income", order=order)
    plt.title("Records by Income Category")
    plt.xlabel("Income category")
    plt.ylabel("Number of records")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.show()

In [ ]:
if "age" in clean.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(data=clean, x="age", bins=25, kde=True)
    plt.title("Age Distribution")
    plt.xlabel("Age")
    plt.tight_layout()
    plt.show()

In [ ]:
if "hours_per_week" in clean.columns and "income" in clean.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=clean, x="income", y="hours_per_week")
    plt.title("Weekly Work Hours by Income Category")
    plt.xlabel("Income category")
    plt.ylabel("Hours per week")
    plt.tight_layout()
    plt.show()

In [ ]:
numeric_cols = clean.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) >= 2:
    corr = clean[numeric_cols].corr(numeric_only=True)
    plt.figure(figsize=(9, 6))
    sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
    plt.title("Correlation Heatmap of Numeric Variables")
    plt.tight_layout()
    plt.show()
    display(corr.round(2))

## 7. Outlier review (IQR method)

In [ ]:
def iqr_outlier_summary(series):
    s = pd.to_numeric(series, errors="coerce").dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return pd.Series({
        "Q1": q1, "Q3": q3, "IQR": iqr,
        "lower_bound": low, "upper_bound": high,
        "potential_outlier_count": ((s < low) | (s > high)).sum()
    })

outlier_cols = [c for c in ["age", "hours_per_week", "capital_gain", "capital_loss"] if c in clean.columns]
if outlier_cols:
    display(pd.DataFrame({c: iqr_outlier_summary(clean[c]) for c in outlier_cols}).T.round(2))

## 8. Key findings and interpretation

In [ ]:
print("Dataset rows:", len(df))
print("Rows after duplicate removal:", len(clean))
print("Columns:", clean.shape[1])

if "income" in clean.columns:
    print("\nIncome counts:")
    display(clean["income"].value_counts(dropna=False).to_frame("count"))

for col in ["age", "hours_per_week"]:
    if col in clean.columns:
        print(f"\n{col} summary:")
        print(clean[col].describe().round(2))

if "income" in clean.columns and "hours_per_week" in clean.columns:
    print("\nMedian hours per week by income category:")
    display(clean.groupby("income")["hours_per_week"].median().to_frame("median_hours"))

### Interpretation notes
- Use the actual counts and statistics printed above when writing findings.
- The income distribution indicates how the target categories are represented in this dataset.
- Age and work-hour charts show distributions and possible unusual values.
- Correlation is an association measure, not evidence of causation.
- IQR flags are prompts for investigation; do not automatically remove outliers without context.
- Demographic attributes can be sensitive. Avoid using descriptive patterns to make unfair individual-level decisions.

## 9. Conclusion

This notebook demonstrates an end-to-end exploratory data analysis workflow on a public real-world dataset: loading, inspection, quality checks, preprocessing, statistical summaries, visualization, and interpretation. Before submitting, run all cells, confirm charts and outputs display correctly, and use the actual results from your run in the written report.